### Entraînement du modèle
Le modèle pour la classification gauche-droite des messages en français est composé de deux parties. D'abord, on utilise un SentenceBERT pour otbenir les embeddings de chaque message. Puis on applique un XGBClassifier sur ces embeddings pour obtenir une classification gauche-droite.

Le SentenceBERT est déjà entraîné. Par contre, nous devons entrainer le XGBClassifier, et en particulier ses hyperparamètres. Ce notebook utilise les messages annotés en français pour choisir les meilleurs hyperparamètres, et évalue, puis sauvegarde, le modèle final.

In [ ]:
import numpy as np
import pandas as pd
import optuna
import torch

from sentence_transformers import SentenceTransformer

from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import langid
import re
import unicodedata
import html

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 1. HTML entities (&gt etc.)
    text = html.unescape(text)

    # 2. Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # 3. Fix escaped apostrophes (IMPORTANT)
    text = text.replace("\\'", "'")

    # 4. Remove leftover backslashes
    text = text.replace("\\", "")

    # 5. Fix non-breaking spaces
    text = text.replace("\xa0", " ")

    # 6. Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

### Récupération des messages annotés en français

In [ ]:
df = pd.read_csv("../annotations_left-right/sample_annotated_left-right.csv")
df = df[df['annotation']!='Unclassifiable']

langid.set_languages(['en', 'fr'])
df["language"] = df["text"].apply(lambda x: langid.classify(x)[0])
df = df[df['language']=='fr']

df['num_annotation'] = [0 if l=='Left' else 1 for l in df['annotation']]
y = df['num_annotation'].values.astype(np.int32)
len(y)

### Calcul des embeddings avec SentenceBERT

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-mpnet-base-v2"
)

sbert = SentenceTransformer(MODEL_NAME,
    device=device)

In [ ]:
texts = df['text'].astype(str).apply(clean_text).tolist()

X = sbert.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)


print("Embeddings shape:", X.shape)

### Recherche des meilleurs hyperparamètres
On utilise la librairie optuna, qui ne fait ni une grid search ni une random search, mais qui implémente des algorithmes de recherche dans l'espace des hyperparamètres. Pour s'assurer de la robustesse des résultats, pour chaque tirage d'hyperparamètres, on évalue le XGBClassifier par une cross-validation à 5 passes.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

def objective(trial):

    n_neg = np.sum(y == 0)
    n_pos = np.sum(y == 1)

    imbalance_ratio = n_neg / n_pos

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            800
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.5,
            log=True
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            5
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.1,
            10
        ),

        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight",
            max(0.5, imbalance_ratio * 0.5),
            imbalance_ratio * 2
        ),

        "objective": "binary:logistic",

        "eval_metric": "logloss",

        "tree_method": "hist",

        "random_state": 42
    }

    threshold = trial.suggest_float(
        "threshold",
        0.1,
        0.9
    )

    fold_scores = []

    for train_idx, valid_idx in cv.split(X, y):

        X_train, X_valid = X[train_idx], X[valid_idx]

        y_train, y_valid = y[train_idx], y[valid_idx]

        model = XGBClassifier(**params)

        model.fit(X_train, y_train)

        y_prob = model.predict_proba(X_valid)[:, 1]

        y_pred = (
            y_prob >= threshold
        ).astype(int)
        
        precision = f1_score( # On veut maximiser la précision.
            y_valid,
            y_pred,
            average="macro",
            #zero_division=0
            )

        fold_scores.append(precision)

    return np.mean(fold_scores)

In [ ]:
optuna.logging.set_verbosity(
    optuna.logging.WARNING
)

study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=500,
    show_progress_bar=True
)

### Evaluation sur les meilleurs paramètres

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
best_params = {'n_estimators': 579, 'max_depth': 10, 'learning_rate': 0.18767857646049085, 'subsample': 0.651887963753939, 'colsample_bytree': 0.6208042297903424, 'min_child_weight': 9, 'gamma': 1.0490747434777725, 'reg_alpha': 2.8491942460956086, 'reg_lambda': 8.068984282856123, 'scale_pos_weight': 2.9663750683596453, 'threshold': 0.5289310689146067}
best_treshold = best_params.pop('threshold')
final_model = XGBClassifier(
    **best_params,

    objective="binary:logistic",

    eval_metric="logloss",

    tree_method="hist",

    random_state=42
)

final_model.fit(X_train, y_train)

def predict_with_threshold(model, X, threshold=0.5):

    proba = model.predict_proba(X)[:, 1]

    return (proba >= threshold).astype(int)

y_pred = predict_with_threshold(final_model, X_test, best_treshold)

Calcul des métriques de performance

In [ ]:
# Métriques par classe
precision_per_class = precision_score(y_test, y_pred, average=None)
recall_per_class = recall_score(y_test, y_pred, average=None)
f1_per_class = f1_score(y_test, y_pred, average=None)

# Support par classe
support_class_0 = len(y_test) - sum(y_test)
support_class_1 = sum(y_test)

# Moyennes macro
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

# Affichage
for i in range(2):
    support = support_class_0 if i == 0 else support_class_1

    print(f"Classe {i}")
    print(f"Precision : {precision_per_class[i]:.4f}")
    print(f"Recall    : {recall_per_class[i]:.4f}")
    print(f"F1-score  : {f1_per_class[i]:.4f}")
    print(f"Support   : {support}")
    print()

print("=== Moyennes macro ===")
print(f"Precision macro : {precision_macro:.4f}")
print(f"Recall macro    : {recall_macro:.4f}")
print(f"F1 macro        : {f1_macro:.4f}")

print("\n=== Accuracy ===")
print(f"Accuracy : {accuracy:.4f}")

### Sauvegarde du XGBClassifier

In [ ]:
final_model.save_model("XGBmodel.json") 
# On reporte le best_threshold à la main dans le notebook d'inférence